# SignSage v2 — Kaggle training run (multi-session)

Runs `research/training/train_sign_to_text.py` (corrected post-352868924 config).
Read `research/training/READINESS.md` first. Requires a Kaggle GPU + `HF_TOKEN` secret.
Each step below is one cell; outputs are the run record.

> **V2 EXPERIMENT — NOT PRODUCTION.** A trained v2 checkpoint is an evaluation
> candidate only. Model promotion requires a separate evaluation review against
> the repository's promotion criteria. Nothing in this notebook sets
> `SIGNSAGE_MODEL_PROMOTED` or touches production serving.

**How to run:** do NOT press Run All in the interactive editor for the long run.
Save the notebook, then **Save Version -> Save & Run All** with **GPU enabled**, and let the
batch execution run top-to-bottom. Do not run the training cell manually while the batch job is running. Batch execution is independent of the editor UI: Kaggle's "Are you still there?" idle warning (after ~40 min without editing) applies to the interactive editor only and does not pause or kill a Save & Run All batch job -- never fake activity to keep the editor awake.

**Multi-session flow (12h limit):** Session 1 = restore-check -> provision -> preflight ->
audit -> **measure** (feasibility + auto subset) -> **certify-pilot** (if subset) -> **prepare** (pose cache + tokenizer, resumable) -> overfit gate -> begin training.
Session 2+ = attach prior output as input dataset -> restore cell reuses cache/tokenizer/
checkpoint -> `--resume` continues training. `/kaggle/working` is ephemeral: nothing
persists unless you **Save Version** (persist cell) and re-attach it next session.

Flow: setup -> restore -> provision/preflight -> audit -> **measure** (feasibility verdict)
-> **certify-pilot** (only when a subset was auto-selected)
-> **prepare** (`tokenizer,prepare`, cache mode, only on GO/certified pilot) -> **train**
(`tokenizer,overfit,train`, 95% reachable-reference gate must pass)
-> held-out evaluate + export -> persist -> adapter check. Failed/interrupted
prepare or training stages skip later model stages but still reach persistence.


**Required before Save & Run All:** Secrets `GITHUB_TOKEN` + `HF_TOKEN` (Add-ons > Secrets);
accelerator **GPU T4 x2**; input dataset = prior session output containing `s2t_v2/`
(set `PRIOR_OUTPUT_SLUG` env only when auto-detect is ambiguous); optional iSign CSV
dataset (`ISIGN_DATASET_SLUG`) or HF download fallback. Safe to re-run any time:
restore-check, provision, preflight, audit, measure. Depend on restored artifacts:
prepare (resumes cache), training (needs cache/tokenizer), evaluate/export/adapter
(need successful training). Only a saved output version persists `/kaggle/working`.


In [ ]:
!pip install -q pose-format jiwer sacrebleu rouge-score sentencepiece huggingface_hub requests tqdm pandas zstandard


In [ ]:
# Requires: provisioning + preflight cells above (HF_TOKEN in subprocess env).
# Private-repo clone: GITHUB_TOKEN via Kaggle Secrets/env, used ONLY for
# this clone through a one-shot HTTP Authorization header. The persisted
# remote stays clean (https://github.com/therandomuser03/signsage.git) and
# the token is never printed, never in history, never in .git/config.
import base64
import json as _json
import os
import subprocess
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/therandomuser03/signsage.git"
DEST = Path("/kaggle/working/signsage")
TRAIN_SCRIPT = "research/training/train_sign_to_text.py"

token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
print("GITHUB_TOKEN available:", bool(token))
if not token:
    raise RuntimeError("GITHUB_TOKEN is missing: add it under the Kaggle notebook Add-ons > Secrets, then re-run this cell.")

# Preflight: confirm the token can actually read the private repo.
try:
    _req = urllib.request.Request(
        "https://api.github.com/repos/therandomuser03/signsage",
        headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(_req, timeout=20) as _r:
        _info = _json.loads(_r.read().decode())
except Exception as exc:
    raise RuntimeError("Cannot access private repo therandomuser03/signsage with GITHUB_TOKEN (invalid/expired token or missing repo permission).") from exc
print("private repo accessible:", _info.get("full_name"), "| private:", _info.get("private"))

PINNED_SHA = "43ed3572d7854de4aa8c39ef27b21a8632c51f64"

if not (DEST / ".git").exists():
    _cred = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    try:
        subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {_cred}",
                        "clone", REPO_URL, str(DEST)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Clone of private repo failed (auth/network).") from exc
    finally:
        del _cred
del token

# Pin the exact verified commit; fail closed on any mismatch (no silent drift).
try:
    subprocess.run(["git", "-C", str(DEST), "checkout", PINNED_SHA],
                   check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(f"Cannot checkout pinned commit {PINNED_SHA}.") from exc
_resolved = subprocess.run(["git", "-C", str(DEST), "rev-parse", "HEAD"],
                           capture_output=True, text=True, check=True).stdout.strip()
print("repo commit:", _resolved)
if _resolved != PINNED_SHA:
    raise RuntimeError(f"Commit pin mismatch: resolved {_resolved} != pinned {PINNED_SHA}.")

# Post-conditions: clean remote URL, expected training file present.
_remote = subprocess.run(["git", "-C", str(DEST), "remote", "get-url", "origin"],
                         capture_output=True, text=True, check=True).stdout.strip()
assert _remote == REPO_URL, f"remote URL polluted: {_remote!r}"
assert (DEST / TRAIN_SCRIPT).is_file(), f"missing {TRAIN_SCRIPT} in cloned repo"
print("clone OK; remote clean; training script present")

OUT = "/kaggle/working/s2t_v2"
os.makedirs(OUT, exist_ok=True)
print("repo + out ready (run the provisioning cell next)")


In [ ]:
# SESSION RESTORE (run FIRST every session): /kaggle/working is ephemeral.
# If a previous session's output was attached as an input dataset, copy it back
# and report exactly what will be reused. First session: nothing found -> fresh.
# To point at a specific prior output, set PRIOR_OUTPUT_SLUG to its dataset slug
# (the folder name under /kaggle/input) before running this cell.
import json
import os
import shutil
from pathlib import Path

OUT = "/kaggle/working/s2t_v2"
_prepare_succeeded = None
_training_succeeded = None
_pipeline_error = None
restored = None
_marker = Path(OUT) / "checkpoints" / "signsage_transformer_recovery.pt"
if not _marker.is_file():
    cands = []
    slug = os.environ.get("PRIOR_OUTPUT_SLUG", "").strip()
    if slug:
        cands.append(Path("/kaggle/input") / slug / "s2t_v2")
    else:
        if Path("/kaggle/input").exists():
            cands += sorted(Path("/kaggle/input").glob("*/s2t_v2"))
        _valid = [_c for _c in cands if ((_c / "checkpoints").is_dir() or (_c / "tokenizer").is_dir()
                or (_c / "pose_cache").is_dir())]
        if len(_valid) > 1:
            raise RuntimeError("ambiguous restore candidates %s; set PRIOR_OUTPUT_SLUG to exactly one" % _valid)
    for _c in cands:
        if ((_c / "checkpoints").is_dir() or (_c / "tokenizer").is_dir()
                or (_c / "pose_cache").is_dir()):
            shutil.copytree(_c, OUT, dirs_exist_ok=True)
            restored = str(_c)
            break
    if slug and restored is None:
        raise RuntimeError("PRIOR_OUTPUT_SLUG=%r did not resolve to a usable s2t_v2 output under /kaggle/input" % slug)
print("restored from:", restored or "(fresh session -- nothing to reuse yet)")
_repo = Path("/kaggle/working/signsage")
if (_repo / ".git").exists():
    try:
        import subprocess as _sp
        _sha = _sp.run(["git", "-C", str(_repo), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        print("repo commit:", _sha or "UNKNOWN")
    except Exception as exc:
        print(f"repo commit: UNKNOWN ({type(exc).__name__})")
else:
    print("repo commit: UNKNOWN (clone cell has not run yet this session)")

# Status report: exactly what the later cells will reuse vs rebuild.
_man = Path(OUT) / "pose_cache" / "manifest.json"
if _man.is_file():
    try:
        _m = json.loads(_man.read_text())
        print(f"pose cache: {_m.get('n', '?')} samples, complete={_m.get('complete')}")
        if _m.get('complete') is not True:
            print("WARNING: pose cache INCOMPLETE (not complete) -- prepare cell will resume/rebuild")
    except ValueError:
        print("pose cache: manifest unreadable (will be rebuilt)")
else:
    print("pose cache: ABSENT (prepare cell will build it)")
_tok = Path(OUT) / "tokenizer" / "signsage_en_sp.model"
print("tokenizer:", "REUSE " + str(_tok) if _tok.is_file() else "ABSENT (prepare cell will train it)")
print("token cache:", "REUSE" if (Path(OUT) / "tokenizer" / "tokens.npz").is_file() else "ABSENT")
_rec = Path(OUT) / "checkpoints" / "signsage_transformer_recovery.pt"
if _rec.is_file():
    try:
        import torch
        _r = torch.load(_rec, map_location="cpu", weights_only=False)
        print(f"recovery checkpoint: REUSE (epoch={_r.get('epoch')}, "
              f"best={float(_r.get('best', float('nan'))):.4f}, history={len(_r.get('history', []))})")
    except Exception as exc:
        print(f"recovery checkpoint: present but unreadable ({type(exc).__name__})")
else:
    print("recovery checkpoint: ABSENT (training starts at epoch 1)")
_best = Path(OUT) / "checkpoints" / "signsage_transformer_full.pt"
print("best checkpoint:", "REUSE " + str(_best) if _best.is_file() else "ABSENT")

# Fail-closed reuse check: restored artifacts must match the CURRENT training
# code before later cells reuse them. Pose-cache version, tokenizer files, and
# token-cache metadata are verified here; per-split fingerprints are enforced
# again at use time by the training script (it rebuilds the token cache on any
# fingerprint mismatch instead of silently reusing it). A failed restore never
# becomes a costly full-data run on its own: prepare stays budget-guarded and
# training refuses on a FAIL feasibility verdict unless --force is passed.
try:
    import sys as _sys_v
    _repo_src = _repo / "research" / "training"
    if str(_repo_src) not in _sys_v.path:
        _sys_v.path.insert(0, str(_repo_src))
    import train_sign_to_text as _T
    _exp_ver = int(_T.POSE_CACHE_VERSION)
    _man_path = Path(OUT) / "pose_cache" / "manifest.json"
    if _man_path.is_file():
        try:
            _mv = json.loads(_man_path.read_text())
            if int(_mv.get("version", -1)) != _exp_ver:
                print(f"pose cache: REJECTED (manifest version {_mv.get('version')} != code {_exp_ver} -- prepare will rebuild)")
            elif not isinstance(_mv.get("uid_to_shard"), dict) or not _mv.get("uid_to_shard"):
                print("pose cache: REJECTED (manifest has no uid_to_shard entries -- prepare will rebuild)")
            else:
                print(f"pose cache: VERIFIED (version {_exp_ver}, {_mv.get('n')} entries, complete={_mv.get('complete')})")
        except (ValueError, TypeError, AttributeError):
            print("pose cache: REJECTED (manifest unreadable -- prepare will rebuild)")
    else:
        print("pose cache: MISSING (verified absent -- prepare will build it)")
    _tok_model = Path(OUT) / "tokenizer" / "signsage_en_sp.model"
    _tok_npz = Path(OUT) / "tokenizer" / "tokens.npz"
    _tok_meta = Path(OUT) / "tokenizer" / "tokens_meta.json"
    if _tok_model.is_file() and _tok_npz.is_file() and _tok_meta.is_file():
        try:
            _tm = json.loads(_tok_meta.read_text())
            print("tokenizer: VERIFIED (model + tokens.npz + tokens_meta.json present; "
                  f"splits={sorted(_tm.get('splits', {})) or 'unknown'})")
        except ValueError:
            print("tokenizer: REJECTED (tokens_meta.json unreadable -- tokenizer stage will rebuild)")
    elif _tok_model.is_file() or _tok_npz.is_file():
        print("tokenizer: REJECTED (partial files -- tokenizer stage will rebuild; will not silently reuse)")
    else:
        print("tokenizer: MISSING (verified absent -- tokenizer stage will build it)")
except Exception as _ve:
    print(f"artifact verification deferred ({type(_ve).__name__}: clone cell has not run yet or training code not importable; "
          "prepare/training stages re-verify before reuse)")



In [ ]:
# Dataset provisioning: resolve iSign_v1.1.csv from (1) an existing working
# copy, (2) an attached Kaggle dataset (set ISIGN_DATASET_SLUG if attached
# under a custom slug), or (3) a Hugging Face download. Fails early with a
# clear error when none is available -- training never starts without data.
# Also guarantees HF_TOKEN is exported for pose streaming (--pose-source
# remote always needs it): subprocesses inherit os.environ, plain Python
# locals do not. The token itself is never printed.
import os
from pathlib import Path

def _resolve_csv():
    cands = [Path("/kaggle/working/isign/iSign_v1.1.csv")]
    slug = os.environ.get("ISIGN_DATASET_SLUG", "").strip()
    if slug:
        cands.append(Path("/kaggle/input") / slug / "iSign_v1.1.csv")
    if Path("/kaggle/input").exists():
        cands += sorted(Path("/kaggle/input").glob("*/iSign_v1.1.csv"))
    for _p in cands:
        if _p.is_file():
            return _p
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = os.environ.get("HF_TOKEN") or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("HF_TOKEN Kaggle Secret (or env) is required to download iSign from Hugging Face.") from exc
    if not _tok:
        raise RuntimeError("HF_TOKEN is empty: attach the CSV as a Kaggle dataset or set the HF_TOKEN secret.")
    os.environ["HF_TOKEN"] = _tok
    del _tok
    from huggingface_hub import hf_hub_download
    _dest = Path("/kaggle/working/isign")
    _dest.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(repo_id="Exploration-Lab/iSign", filename="iSign_v1.1.csv",
                                repo_type="dataset", token=os.environ["HF_TOKEN"], local_dir=str(_dest)))

CSV_PATH = str(_resolve_csv())
print("iSign CSV:", CSV_PATH)
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient as _USC
        _t = _USC().get_secret("HF_TOKEN")
        if _t:
            os.environ["HF_TOKEN"] = _t
        del _t
    except Exception as exc:
        raise RuntimeError("HF_TOKEN is required for remote pose streaming (--pose-source remote): set the HF_TOKEN Kaggle Secret, then re-run.") from exc
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is empty: set the HF_TOKEN Kaggle Secret for remote pose streaming, then re-run.")
print("HF_TOKEN available for pose streaming:", True)


In [ ]:
# Preflight (re-run any time): prove a REAL subprocess inherits HF_TOKEN
# before any expensive command. Reports availability + length ONLY --
# the token value is never printed, written, or embedded anywhere.
import os
import subprocess
import sys

_tok = os.environ.get("HF_TOKEN")
print("HF_TOKEN available:", bool(_tok))
if not _tok:
    raise RuntimeError("HF_TOKEN missing in kernel env: re-run the provisioning cell above (it pulls the Kaggle Secret into os.environ), then re-run this cell.")
_probe = subprocess.run(
    [sys.executable, "-c",
     "import os; _t=os.environ.get('HF_TOKEN'); "
     "print('subprocess HF_TOKEN available:', bool(_t)); "
     "print('token length:', len(_t) if _t else 0)"],
    capture_output=True, text=True)
print(_probe.stdout.strip())
if _probe.returncode != 0 or "subprocess HF_TOKEN available: True" not in _probe.stdout:
    raise RuntimeError("subprocess cannot see HF_TOKEN: restart the kernel, run cells top-to-bottom (provisioning, then this cell), and retry.")
print("preflight OK: training subprocesses will inherit HF_TOKEN")


In [ ]:
# Pose-source preflight (cheap, re-run any time): proves the remote pose
# archive is reachable via HTTP Range reads AND that one REAL pose member
# decodes end-to-end -- before any expensive stage. Writes
# /kaggle/working/s2t_v2/pose_preflight.json and exits; no training starts.
# Requires: provisioning + HF_TOKEN preflight cells above.
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --preflight-only

In [ ]:
!cd /kaggle/working/signsage && python research/training/audit_dataset.py --csv {CSV_PATH} --out /kaggle/working/s2t_v2/dataset_audit.json

In [ ]:
# MEASURE FIRST (dry-run, ~2,000 samples, minutes): decides EVERYTHING downstream.
# Runs the EXACT V2 preprocessing path on a deterministic stride sample, zstd-checks
# it, times tokenizer training on the FULL training corpus (persisted + reused by
# later stages, even for pilots), benchmarks real training-step and decode
# throughput, and writes reports:
#   /kaggle/working/s2t_v2/measure_report.json  (zstd FIT/BORDERLINE/DOES_NOT_FIT)
#   /kaggle/working/s2t_v2/feasibility.json     (quota verdict vs 13h34m / 11.5h target)
#   /kaggle/working/s2t_v2/auto_subset.json     (largest safe per-split subset, or 0)
# Creates NO pose cache. Run this BEFORE any prepare: the training cell REFUSES to
# start on a FAIL verdict unless you pass "--force" (quota WILL burn).
# --persistence-target-gib is YOUR measured budget (see df output in persist cell),
# never assumed: omit it and the zstd verdict stays UNKNOWN-no-target.
import json as _json
import math as _math
import subprocess
import sys

OUT_DIR = "/kaggle/working/s2t_v2"

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", OUT_DIR,
    "--stages", "measure",
    "--measure-samples", "2000",
]

# Reuse guard: the 46-min remote measurement already produced valid
# measure_report.json / feasibility.json. Never re-run it just because the
# report formatting failed; reuse existing artifacts when present and valid.
def _have_valid_reports():
    try:
        _r = _json.load(open(OUT_DIR + "/measure_report.json"))
        _f = _json.load(open(OUT_DIR + "/feasibility.json"))
        _a = _json.load(open(OUT_DIR + "/dataset_audit.json"))
        _fetch = float(_r["fetch_prep_per_sample_s"])
        _steps = float(_r["benchmarks"]["train_steps_per_sec"])
        _dec = float(_r["benchmarks"]["decode_hyps_per_sec"])
        float(_r["compression_ratio"])
        float(_r["mean_zero_padding_fraction"])
        float(_f["projected_total_hours"])
        float(_f["target_hours"])
        if not (_math.isfinite(_fetch) and _fetch > 0):
            return False
        if not (_math.isfinite(_steps) and _steps > 0):
            return False
        if not (_math.isfinite(_dec) and _dec > 0):
            return False
        if _f.get("verdict") not in ("PASS", "FAIL"):
            return False
        _split = _a["split"]
        _counts = {"train": int(_split["train_rows"]),
                   "val": int(_split["val_rows"]),
                   "test": int(_split["test_rows"])}
        if _r.get("split_counts") != _counts:
            return False
        if _f["breakdown_hours"]["epochs"] != 12:
            return False
        return True
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return False

_reuse_reports = _have_valid_reports()

if _reuse_reports:
    print("Reusing existing measure_report.json / feasibility.json (no re-measure).")
else:
    print("========================================")
    print("MEASURING FEASIBILITY (dry-run, bounded)")
    print("========================================")
    result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
    if result.returncode != 0:
        raise RuntimeError("Measure failed.")

_rep = _json.load(open(OUT_DIR + "/measure_report.json"))
print()
print("true_len: min=%(min)s median=%(median).0f mean=%(mean).1f p90=%(p90).0f p95=%(p95).0f max=%(max)s" % _rep["true_len"])
print("compression ratio: %.2fx (dense-data ratio %.2fx, zero-padding fraction %.1f%%)" % (
    _rep["compression_ratio"], _rep["dense_data_compression_ratio"],
    100 * _rep["mean_zero_padding_fraction"]))
print("projected full 123,779: %.1f GiB" % _rep["projected_compressed_gib_full_123779"])
print("by split: " + ", ".join("%s=%.1f GiB" % (k, v) for k, v in _rep["projected_compressed_gib_by_split"].items()))
print("zstd verdict: %s" % _rep["verdict"])
print("fetch+prep: %.3f sec/sample (%.1f samples/sec)" % (
    _rep["fetch_prep_per_sample_s"], 1 / _rep["fetch_prep_per_sample_s"]))
print("train steps: %.2f/sec | decode: %.1f hyps/sec | tokenizer: %.0fs (full corpus, persisted, reused)" % (
    _rep["benchmarks"]["train_steps_per_sec"], _rep["benchmarks"]["decode_hyps_per_sec"],
    _rep["tokenizer_hours"] * 3600))

_feas = _json.load(open(OUT_DIR + "/feasibility.json"))
print()
print("========================================")
print("V2 RESOURCE FEASIBILITY")
print("========================================")
print("Remaining Kaggle budget: %.2fh" % _feas["quota_hours"])
print("Target maximum runtime: %.2fh" % _feas["target_hours"])
print("Estimated epoch: %.2f hours" % _feas["breakdown_hours"]["per_epoch"])
print("Estimated total (%d epochs): %.2f hours" % (
    _feas["breakdown_hours"]["epochs"], _feas["projected_total_hours"]))
print("FEASIBILITY: %s" % _feas["verdict"])
print("========================================")


_LOCAL_FETCH_S = 0.02  # conservative upper bound for local shard reads;
# measured 2026-10-08: ~6ms/sample (158/s) through the full V2 path on
# 160-frame worst-case blobs. Remote 1.18s/sample is paid ONCE in prepare;
# post-prepare gate/train/val/diversity/eval read LOCAL shards (no HTTP).


def _project_subset_hours(n_per_split, fetch_s, steps_s, dec_s, tok_h, epochs=12,
                           local_s=_LOCAL_FETCH_S):
    # Cache-mode mirror of compute_feasibility(data_mode="cache") in
    # train_sign_to_text.py, plus a prepare-time estimate (~4x effective
    # parallelism of --jobs 8). Gate/diversity loaders memoize (64/512 reads
    # once total), so the gate data term is 64*local -- never 600*64*remote.
    # Couplings that MUST match the prepare/training cells below: epochs=12,
    # batch=64, --data-mode cache, --cache-budget-gib 10.
    train_steps = _math.ceil(n_per_split / 64)
    val_steps = _math.ceil(n_per_split / 64)
    epoch_data_h = (n_per_split * 2) * local_s / 3600  # train+val, LOCAL shards
    epoch_comp_h = (train_steps + val_steps / 3) / steps_s / 3600
    epoch_h = epoch_data_h + epoch_comp_h
    prepare_h = (3 * n_per_split * fetch_s) / 4 / 3600  # remote fetch, ONCE
    gate_h = (64 * local_s + 1200 * 4 / steps_s) / 3600  # memoized, LOCAL;
    # 1200 mirrors GATE_STEPS in train_sign_to_text.py (worst case; early break usual)
    div_h = 512 / dec_s / 3600
    eval_h = n_per_split * local_s / 3600 + n_per_split / dec_s / 3600 + 0.2
    total_h = prepare_h + tok_h + gate_h + epochs * (epoch_h + div_h) + eval_h + 0.5
    return total_h, prepare_h


_FP32_BYTES_PER_SAMPLE = 160 * 1629 * 4  # exact V2 tensor shape, not an estimate
_selection = {"subset": 0, "projected_total_hours": None, "prepare_hours": None,
              "cache_gib_heuristic": None, "target_hours": _feas["target_hours"],
              "basis": "full-data feasibility verdict"}
if _feas["verdict"] == "FAIL":
    _fetch_s = _rep["fetch_prep_per_sample_s"]
    _steps_s = _rep["benchmarks"]["train_steps_per_sec"]
    _dec_s = _rep["benchmarks"]["decode_hyps_per_sec"]
    _tok_h = _rep.get("tokenizer_hours", 0.0) or 0.0
    _target_h = _feas["target_hours"]
    _measurable = all(_math.isfinite(v) and v > 0 for v in (_fetch_s, _steps_s, _dec_s))
    if _measurable:
        for _sz in (3000, 2500, 2000, 1500, 1000, 500):
            _proj, _prep = _project_subset_hours(_sz, _fetch_s, _steps_s, _dec_s, _tok_h)
            # Disk sanity heuristic: raw .pose cache ~= fp32 tensor bytes x 1.25
            # headroom (historical raw/fp32 ~= 1.19). The prepare cell's budget
            # guard stays authoritative and refuses safely if this is wrong.
            _cache_gib = 3 * _sz * _FP32_BYTES_PER_SAMPLE * 1.25 / (1024 ** 3)
            if _proj <= _target_h - 1.5 and _cache_gib <= 10.0:
                _selection.update(subset=_sz, projected_total_hours=round(_proj, 2),
                                  prepare_hours=round(_prep, 2),
                                  cache_gib_heuristic=round(_cache_gib, 2),
                                  basis="cache-mode measured/local rates, 1.5h margin, 10 GiB budget heuristic")
                break
    with open(OUT_DIR + "/auto_subset.json", "w") as _fh:
        _json.dump(_selection, _fh, indent=1)
    if _selection["subset"]:
        print()
        print("AUTO-SELECTED SUBSET: %d per split (3*%d=%d total)" % (
            _selection["subset"], _selection["subset"], 3 * _selection["subset"]))
        print("  Projected runtime: %.2fh (target <= %.2fh, margin ~%.2fh)" % (
            _selection["projected_total_hours"], _target_h,
            _target_h - _selection["projected_total_hours"]))
        print("  Est. prepare: %.2fh | est. cache: %.1f GiB (heuristic)" % (
            _selection["prepare_hours"], _selection["cache_gib_heuristic"]))
        print("NEXT: run the CERTIFY PILOT cell to scope feasibility to this pilot.")
    else:
        print()
        print("NO SAFE SUBSET FOUND -- check measurements; do NOT start training.")
    print("Full data FAILs quota: do NOT run prepare/train on all 123,779 samples.")
else:
    with open(OUT_DIR + "/auto_subset.json", "w") as _fh:
        _json.dump(_selection, _fh, indent=1)
    print()
    print("Full dataset fits within quota. No subset needed; skip the CERTIFY cell.")


In [ ]:
# CERTIFY PILOT (run ONLY when the measure cell auto-selected a subset).
# Re-runs the bounded measure WITH --subset N so feasibility.json is scoped to the
# pilot itself (pose work + benches on pilot rows; the tokenizer is reused from the
# full corpus, ~free). The training cell enforces THIS verdict: PASS proceeds with
# --subset N, FAIL stops here. Full-scope reports are backed up first (*_full.json).
# If this raises FAIL, do NOT pass --force without a smaller manual subset.
import json as _json
import shutil
import subprocess
import sys

OUT_DIR = "/kaggle/working/s2t_v2"

try:
    _sel = _json.load(open(OUT_DIR + "/auto_subset.json"))
    subset_size = int(_sel.get("subset", 0))
except (OSError, ValueError, TypeError, AttributeError):
    subset_size = 0

if not subset_size:
    print("No pilot subset selected (full data fits, or no safe subset). Skipping certify.")
else:
    import math as _math_cert
    import os as _os_cert
    for _name in ("measure_report.json", "feasibility.json"):
        _src = OUT_DIR + "/" + _name
        _dst = OUT_DIR + "/" + _name.replace(".json", "_full.json")
        # Never clobber the full-scope backup with pilot-scoped reports.
        if not _os_cert.path.exists(_dst):
            try:
                shutil.copy(_src, _dst)
            except OSError:
                pass
    cmd = [
        sys.executable,
        "research/training/train_sign_to_text.py",
        "--csv", CSV_PATH,
        "--pose-source", "remote",
        "--out", OUT_DIR,
        "--stages", "measure",
        "--measure-samples", "2000",
        "--subset", str(subset_size),
        "--data-mode", "cache",
    ]
    print("========================================")
    print("CERTIFYING PILOT (subset=%d per split, cache mode, bounded)" % subset_size)
    print("========================================")
    result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
    if result.returncode != 0:
        raise RuntimeError("Pilot certify-measure failed.")
    _feas = _json.load(open(OUT_DIR + "/feasibility.json"))
    try:
        _proj = float(_feas.get("projected_total_hours", float("nan")))
    except (TypeError, ValueError):
        _proj = float("nan")
    print()
    print("PILOT FEASIBILITY: %s" % _feas["verdict"])
    if _feas["verdict"] == "PASS" and _math_cert.isfinite(_proj):
        print("Pilot projected total: %.2f hours (target <= %.2fh)" % (
            _proj, _feas["target_hours"]))
        print("PILOT CERTIFIED: prepare + training cells will use --subset %d." % subset_size)
    elif _feas["verdict"] == "FAIL" and not _math_cert.isfinite(_proj):
        # Expected pre-prepare: cache-mode prepare cost is unpriced until the
        # 200-sample projection inside PREPARE. Proceed there; the training
        # cell recomputes a finite cache verdict once the cache is COMPLETE.
        print("Pilot scoped to cache mode; prepare cost unpriced (expected before PREPARE).")
        print("NEXT: run PREPARE, then TRAINING (finite verdict recomputed post-prepare).")
    else:
        raise RuntimeError(
            "PILOT NOT CERTIFIED: pilot scope still exceeds quota (%.2fh). " % _proj +
            "Shrink --subset manually (edit auto_subset.json) or STOP.")


In [ ]:
# PREPARE (Session 1; resumable -- safe to re-run after interruption).
# Requires: measure cell says GO (full data), or the CERTIFY PILOT cell certified
# a pilot subset. The scope is read from /kaggle/working/s2t_v2/auto_subset.json
# (written by measure/certify), so this cell and the training cell always use the
# SAME scope -- never pass --subset by hand here.
# --data-mode cache: materializes pose_cache/ from remote ONCE (sharded ZIPs of
# verbatim .pose bytes; processed fp32 tensors would be ~120 GiB and are
# deliberately NOT cached) and trains the tokenizer + token cache (reused if the
# measure cell already built them). Re-running skips present samples via the
# manifest. --data-mode stream (script default, NOT used here): FULL CACHE
# DISABLED, prepare does nothing and every epoch re-reads remotely.
# BUDGET GUARD: projection uses actual shard bytes after 200 samples, with a
# 25% safety margin; it prints measured KiB/sample + projected GiB + ETA and
# REFUSES (keeps the partial cache, burns no more quota) when the
# projection exceeds --cache-budget-gib or 90% of free disk. Full 123,779 at the
# measured ~1,211 KiB/sample is ~143 GiB and does NOT fit: the auto-selected
# pilot keeps the projection inside YOUR disk budget (see df output in persist).
# The tokenizer is always trained on the FULL training corpus (subset mode
# subsets pose data only -- vocabulary stays full-run consistent).
# MEASURED 2026-10-07: ~1,211 KiB/sample, ~1.2 samples/s sequential (~29h full).
# --jobs 8 fetches in parallel (expect ~4h for whatever subset fits; watch ETA).
import subprocess
import sys

def _auto_subset(path="/kaggle/working/s2t_v2/auto_subset.json"):
    # Single source of truth for pilot scope: auto_subset.json written by the
    # measure cell. Env fallback only (re-running prepare/train standalone).
    import json as _j
    import os as _o
    try:
        return int(_j.load(open(path)).get("subset", 0))
    except (OSError, ValueError, TypeError, AttributeError):
        try:
            return int(_o.environ.get("AUTO_SUBSET", "0"))
        except (ValueError, TypeError):
            return 0

subset_size = _auto_subset()

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", "/kaggle/working/s2t_v2",
    "--stages", "tokenizer,prepare",
    "--data-mode", "cache",
    "--cache-budget-gib", "10",
    "--jobs", "8",
    "--resume",
]

if subset_size > 0:
    cmd += ["--subset", str(subset_size)]

print("========================================")
print("PREPARING POSE CACHE + TOKENIZER (cache mode)")
print("(resumable: interrupt any time, re-run this cell)")
if subset_size > 0:
    print("PILOT SUBSET (from auto_subset.json): %d per split" % subset_size)
print("========================================")

try:
    result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
    _prepare_succeeded = result.returncode == 0
except KeyboardInterrupt:
    _prepare_succeeded = False
    _pipeline_error = "Prepare interrupted; completed shards remain available for Save Version."
except OSError as _e:
    _prepare_succeeded = False
    _pipeline_error = "Prepare process could not start: %s" % type(_e).__name__
if _prepare_succeeded is not True:
    _training_succeeded = False
    _pipeline_error = _pipeline_error or "Prepare failed or budget-refused; partial cache is kept."
    print(_pipeline_error)
else:
    print("PREPARE COMPLETE (or already complete -- see 'already complete' above)")


In [ ]:
# V2 TRAINING (refuses on feasibility FAIL unless --force; reads LOCAL CACHE when
# prepare is complete, else streams remotely every epoch in stream mode).
# QUOTA GATE: this cell exits WITHOUT burning GPU when feasibility.json says FAIL.
# Scope comes from /kaggle/working/s2t_v2/auto_subset.json (measure/certify cells):
# full-data GO trains full data; a certified pilot trains WITH --subset N, and the
# gate enforces the PILOT verdict (certify cell). To burn quota anyway, add
# "--force" (quota WILL burn).
# In --data-mode cache the verdict is recomputed with sunk prepare cost once the
# cache is COMPLETE, so a cache-mode FAIL-unknown-prepare can still turn PASS.
# Requires: restore + provision + preflight cells above (HF_TOKEN in subprocess env
# for the first session's remote reads; later sessions train purely from disk).
# Stages: tokenizer (reuse; always the FULL-corpus vocabulary), overfit-gate
# (mandatory, fail-closed, runs on the LOCAL cache), train. The gate MUST pass
# here (64 samples, >=95% exact) before any multi-hour spend; a failure aborts.
# PILOT artifacts are marked PILOT / NOT FOR PRODUCTION in the bundle + manifest.
#
# RESUME ACROSS SESSIONS: before running this cell in session N+1, attach the
# prior session's saved output as an input dataset and run the restore cell
# (set PRIOR_OUTPUT_SLUG if the slug is ambiguous). Then re-run this cell as-is.

import json as _json
import os
import subprocess
import sys
import time

def _auto_subset(path="/kaggle/working/s2t_v2/auto_subset.json"):
    # Single source of truth for pilot scope: auto_subset.json written by the
    # measure cell. Env fallback only (re-running prepare/train standalone).
    import json as _j
    import os as _o
    try:
        return int(_j.load(open(path)).get("subset", 0))
    except (OSError, ValueError, TypeError, AttributeError):
        try:
            return int(_o.environ.get("AUTO_SUBSET", "0"))
        except (ValueError, TypeError):
            return 0

subset_size = _auto_subset()

cmd = [
    sys.executable,
    "research/training/train_sign_to_text.py",
    "--csv", CSV_PATH,
    "--pose-source", "remote",
    "--out", "/kaggle/working/s2t_v2",
    # DIAGNOSTIC RUN (Option-1 LR test, REVERT after): exactly 2 epochs.
    "--epochs", "2",
    "--max-hours", "7",
    # DIAGNOSTIC RUN (Option-1 LR test, REVERT after): explicit peak LR.
    "--lr", "1e-4",
    "--data-mode", "cache",
    "--stages", "tokenizer,overfit,train",
]

if subset_size > 0:
    cmd += ["--subset", str(subset_size)]

# Split counts come from the audit cell's report, never hardcoded.
_audit_path = "/kaggle/working/s2t_v2/dataset_audit.json"
try:
    _split = _json.load(open(_audit_path))["split"]
    _counts = "\n".join([
        "Train: {:,} samples".format(_split["train_rows"]),
        "Validation: {:,} samples".format(_split["val_rows"]),
        "Test: {:,} samples".format(_split["test_rows"]),
    ])
except (OSError, ValueError, KeyError):
    _counts = ("Train/Validation/Test: UNKNOWN "
               "(dataset_audit.json not found -- run the audit cell first)")

print("========================================")
print("STARTING SIGNSAGE V2 TRAINING")
print("========================================")
print(_counts)
if subset_size > 0:
    print("PILOT MODE: subset=%d per split (3*%d=%d total)" % (
        subset_size, subset_size, 3 * subset_size))
print("Epoch budget: 2 (diagnostic, exactly 2 epochs)")
print("Training wall-clock budget: 7 hours")
# DIAGNOSTIC RUN (Option-1 LR test, REVERT after): fresh start, 2 epochs.
print("DIAGNOSTIC: Epoch budget: 2; Resume: DISABLED (fresh start)")
print("========================================")

started = time.time()
if _prepare_succeeded is not True:
    _training_succeeded = False
    _pipeline_error = _pipeline_error or "Prepare stage not completed; training skipped."
    print("CLASSIFICATION: preparation failure -- training skipped, no GPU burned.")
    print(_pipeline_error)
else:
    # DIAGNOSTIC RUN fresh start: reuse cache/tokenizer restored above, but drop
    # stale epoch-1 checkpoints so optimizer/scheduler state from the collapsed
    # run cannot leak into this run (REVERT with the cmd block above after).
    # Runs ONLY when training will actually proceed, so a skipped run can never
    # delete the only recoverable checkpoints; pose_cache/ + tokenizer/ are kept.
    import pathlib as _pl0
    import shutil as _sh
    _sh.rmtree("/kaggle/working/s2t_v2/checkpoints", ignore_errors=True)
    for _stale in ("COLLAPSED.txt", "gate_report.json"):
        try:
            (_pl0.Path("/kaggle/working/s2t_v2") / _stale).unlink(missing_ok=True)
        except OSError:
            pass
    print("DIAGNOSTIC: stale checkpoints/collapse/gate reports cleared; pose_cache + tokenizer reused; training starts at epoch 1")
    try:
        result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
        _training_succeeded = result.returncode == 0
        if not _training_succeeded:
            _pipeline_error = "Training failed (collapse guard or overfit gate -- see classification below); recovery files remain available for Save Version."
    except KeyboardInterrupt:
        _training_succeeded = False
        _pipeline_error = "Training interrupted; epoch recovery checkpoint remains available."
    except OSError as _e:
        _training_succeeded = False
        _pipeline_error = "Training process could not start: %s" % type(_e).__name__

elapsed = (time.time() - started) / 3600
print("Training runtime: %.2f hours" % elapsed)
if _training_succeeded:
    print("V2 TRAINING COMPLETED SUCCESSFULLY")
else:
    print("TRAINING STATUS:", _pipeline_error)
    import pathlib as _pl
    _collapsed = _pl.Path("/kaggle/working/s2t_v2/COLLAPSED.txt")
    if _collapsed.is_file():
        print("COLLAPSE GUARD FIRED -- training stopped by the diversity guard, not the overfit gate.")
        print(_collapsed.read_text().strip())
        print("A passing overfit gate does NOT imply training succeeded; evaluation/export stay skipped.")
    _gr = _pl.Path("/kaggle/working/s2t_v2/gate_report.json")
    _gate_passed = None
    if _gr.is_file():
        try:
            _gd = _json.loads(_gr.read_text())
        except ValueError:
            _gd = {}
        _gate_passed = _gd.get("passed")
        print("GATE REPORT: passed=%s final=%s failures=%d" % (
            _gd.get("passed"), _gd.get("final"), len(_gd.get("failures", []))))
        for _f in _gd.get("failures", [])[:5]:
            print("  miss uid=%s ref=%r hyp=%r (unconstr=%r)" % (
                _f.get("uid"), _f.get("ref"), _f.get("hyp_constrained"),
                _f.get("hyp_unconstrained")))
    else:
        print("gate_report.json ABSENT (gate never reached its first check)")
    if _collapsed.is_file():
        print("CLASSIFICATION: diversity-collapse failure (collapse guard stopped training).")
    elif _gate_passed is False:
        print("CLASSIFICATION: overfit-gate failure (gate did not reach 95% exact; training aborted before the long run).")
    elif _gate_passed is True:
        print("CLASSIFICATION: training error AFTER a passing overfit gate (a passing gate is NOT training success).")
    else:
        print("CLASSIFICATION: other training failure (see status above; evaluation/export stay skipped).")


In [ ]:
# Final test evaluation + deployment-bundle export (reads LOCAL CACHE).
# Requires: restore + provision cells above.
import json
import os
import subprocess
import sys

if _training_succeeded is not True:
    print("EVALUATION/EXPORT SKIPPED:", _pipeline_error or "training did not pass gate")
else:
    def _auto_subset(path="/kaggle/working/s2t_v2/auto_subset.json"):
        try:
            return int(json.loads(open(path).read()).get("subset", 0))
        except (OSError, ValueError, TypeError, AttributeError):
            return int(os.environ.get("AUTO_SUBSET", "0"))

    subset_size = _auto_subset()
    cmd = [sys.executable, "research/training/train_sign_to_text.py",
           "--csv", CSV_PATH, "--pose-source", "remote",
           "--out", "/kaggle/working/s2t_v2", "--data-mode", "cache",
           "--stages", "evaluate,export"]
    if subset_size > 0:
        cmd += ["--subset", str(subset_size)]
        print("PILOT MODE: subset=%d per split (bundle marked PILOT / NOT FOR PRODUCTION)" % subset_size)
    print("RUNNING FINAL TEST + EXPORT")
    try:
        result = subprocess.run(cmd, cwd="/kaggle/working/signsage")
        if result.returncode != 0:
            _pipeline_error = "Evaluation/export failed with code %d." % result.returncode
            _training_succeeded = False
        else:
            report_path = "/kaggle/working/s2t_v2/evaluation_report.json"
            if not os.path.isfile(report_path):
                _pipeline_error = "Evaluation report missing after successful export command."
                _training_succeeded = False
            else:
                with open(report_path, "r", encoding="utf-8") as f:
                    report = json.load(f)
                print("TEST METRICS:", report.get("test_metrics"))
                print("DIVERSITY:", report.get("diversity"))
                print("QUALITY GATE:", report.get("gate"))
                print("Evaluation report:", report_path)
                print("Deployment bundle:", "/kaggle/working/s2t_v2/signsage_deployment_bundle")
    except KeyboardInterrupt:
        _training_succeeded = False
        _pipeline_error = "Evaluation/export interrupted; run can be resumed from saved checkpoints."
    except OSError as _e:
        _training_succeeded = False
        _pipeline_error = "Evaluation/export process could not start: %s" % type(_e).__name__
    if _pipeline_error:
        print("EVALUATION/EXPORT STATUS:", _pipeline_error)


In [ ]:
# PERSIST (run LAST every session): verify what Save Version will keep, then
# use Kaggle's Save Version button to snapshot /kaggle/working as an output
# dataset. Next session: attach that output as input + set PRIOR_OUTPUT_SLUG
# (or rely on auto-detect in the restore cell). Copying the .ipynb alone
# transfers NOTHING -- only a saved output version persists the cache.
# Secrets check: HF_TOKEN/GITHUB_TOKEN must never appear in the output dir.
import json
import os
from pathlib import Path
# PERSISTENCE DIAGNOSTICS (cheap, run any time): ephemeral-disk usage plus the
# labelled size of the current output/cache. No Kaggle storage limit is assumed
# or hardcoded anywhere: compare these numbers against YOUR measured budget
# (df output below + trial Save Version sizes) before relying on persistence.
import subprocess as _sp
_r = _sp.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print("--- df -h /kaggle/working ---")
print(_r.stdout.strip() or "(df unavailable)")
print("--- current output/cache sizes (persistence check, NOT a limit) ---")

OUT = Path("/kaggle/working/s2t_v2")
total = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())
print(f"output size: {total / 2**30:.2f} GiB in {OUT}")
for _sub in ("pose_cache", "tokenizer", "checkpoints"):
    _d = OUT / _sub
    if _d.is_dir():
        _n = sum(1 for _ in _d.rglob("*") if _.is_file())
        _s = sum(p.stat().st_size for p in _d.rglob("*") if p.is_file()) / 2**30
        print(f"  {_sub}: {_n} files, {_s:.2f} GiB")
_man = OUT / "pose_cache" / "manifest.json"
if _man.is_file():
    try:
        _m = json.loads(_man.read_text())
        print(f"  pose cache complete={_m.get('complete')} n={_m.get('n')}")
    except ValueError:
        print("  pose cache manifest UNREADABLE")
for _name in ("HF_TOKEN", "GITHUB_TOKEN"):
    _hits = [str(p) for p in OUT.rglob("*") if p.is_file()
             and _name in p.name.upper()]
    assert not _hits, f"secret-named file in output: {_hits}"
print("no secret-named files in output dir")
print("pipeline status:", "SUCCESS" if _training_succeeded else (_pipeline_error or "NOT STARTED"))
print()
print("NEXT: Kaggle Save Version (save output), then in a new session attach")
print("that output as input data and run the restore cell (PRIOR_OUTPUT_SLUG")
print("only if auto-detect finds the wrong s2t_v2).")




In [ ]:
# Verify the exported bundle loads through the REAL serving adapter (no notebook state).
if _training_succeeded is not True:
    print("ADAPTER CHECK SKIPPED:", _pipeline_error or "no validated export")
else:
    import os
    import sys
    import numpy as np
    sys.path.insert(0, "/kaggle/working/signsage/backend")
    from app.ml.adapters.sign_to_text_adapter import SignToTextAdapter
    bundle = "/kaggle/working/s2t_v2/signsage_deployment_bundle"
    if not os.path.isdir(bundle):
        _pipeline_error = "Exported bundle missing; check evaluation/export output."
    else:
        adapter = SignToTextAdapter.load(bundle, device="cpu")
        frames = np.random.default_rng(0).normal(size=(25, 543, 3)).astype(np.float32)
        result = adapter.transcribe(frames)
        print("bundle loads + transcribes:", result.accepted, result.reason, repr(result.text[:80]))
        print("Random-input output does NOT measure translation quality.")
        print("Use evaluation_report.json for model quality.")